In [1]:
import os
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import panel as pn
import holoviews as hv
import hvplot.pandas

hv.extension('bokeh')
warnings.filterwarnings('ignore')


# ===== CSS injected INTO each widget's shadow DOM (this is what actually works) =====
WIDGET_CSS = """
:host {
    font-size: 15px;
}
.bk-input-group {
    font-size: 15px !important;
}
.bk-input-group > label,
.bk-slider-title,
label {
    font-size: 15px !important;
    font-weight: 600 !important;
    color: #1a1a1a !important;
    margin-bottom: 6px !important;
}
input.bk-input,
select.bk-input {
    border: 2px solid #2c3e50 !important;
    border-radius: 5px !important;
    font-size: 17px !important;
    padding: 8px 12px !important;
    min-height: 40px !important;
    min-width: 110px !important;
    font-weight: 700 !important;
    color: #1a1a1a !important;
    background-color: #ffffff !important;
}
.bk-spin-wrapper {
    border: 2px solid #2c3e50 !important;
    border-radius: 5px !important;
    background-color: #ffffff !important;
}
.bk-spin-wrapper input.bk-input {
    border: none !important;
}
.bk-spin-btn {
    min-width: 26px !important;
    background-color: #f0f0f0 !important;
}
"""

# Section-header CSS (regular DOM, fine to register globally)
SECTION_HEADER_CSS = """
.section-header {
    background-color: #2c3e50;
    color: #ffffff;
    padding: 12px 20px;
    margin-bottom: 14px;
    border-radius: 4px;
}
.section-header h1,
.section-header h2,
.section-header h3 {
    color: #ffffff !important;
    margin: 0;
    font-weight: 700;
    font-size: 22px !important;
}
"""

# Register the global stylesheet ONCE at module import time, not inside a method
pn.extension(raw_css=[SECTION_HEADER_CSS])


class ToxinPredictionDashboard:
    def __init__(self):
        self.models_loaded = self._load_models()

        self.feature_ranges = {
            'Q':          {'min': -333, 'max': 283,   'initial': 20},
            'Q_1m':       {'min': -300, 'max': 1020,  'initial': 90},
            'SpecCond.':  {'min': 56,   'max': 10834, 'initial': 2910},
            'OrgC':       {'min': 1.4,  'max': 5.5,   'initial': 2.4},
            'OrgN':       {'min': 0.0,  'max': 1.6,   'initial': 0.3},
            'PO4':        {'min': 0.0,  'max': 0.53,  'initial': 0.08},
            'WaterTemp':  {'min': 13,   'max': 26.6,  'initial': 22.63},
            'DO':         {'min': 6.6,  'max': 10.7,  'initial': 8.17},
            'pH':         {'min': 7.0,  'max': 8.8,   'initial': 7.96}
        }

        self.feature_descriptions = {
            'Q':          'Flow (Q, m³/s)',
            'Q_1m':       'Antecedent Flow (Q_1mon, m³/s)',
            'SpecCond.':  'Specific Conductance (µS/cm)',
            'OrgC':       'Dissolved Organic Carbon (DOC, mg/l)',
            'OrgN':       'Dissolved Organic Nitrogen (DON, mg/l)',
            'PO4':        'Dissolved Orthophosphate (PO₄, mg/l)',
            'WaterTemp':  'Water Temperature (°C)',
            'DO':         'Dissolved Oxygen (DO, mg/l)',
            'pH':         'pH'
        }

        self.setup_components()
        self.create_layout_components()

        # CRITICAL: only call update_predictions if models loaded; otherwise we'd crash here
        if self.models_loaded:
            self.update_predictions(None)
        else:
            print("Dashboard created without live predictions — models did not load.")

    def _load_models(self):
        """Cross-platform, Azure-friendly model loading."""
        print("==== DASHBOARD INITIALIZATION ====")
        print(f"Current working directory: {os.getcwd()}")
        try:
            print(f"Directory contents: {os.listdir('.')}")
        except Exception as e:
            print(f"Could not list cwd: {e}")

        # Try multiple potential model paths (Windows local, Linux container, Azure App Service)
        potential_model_paths = [
            (Path('Model') / 'rf_pipeline.joblib',
             Path('Model') / 'xgb_pipeline.joblib'),
            (Path('./Model') / 'rf_pipeline.joblib',
             Path('./Model') / 'xgb_pipeline.joblib'),
            (Path('../Model') / 'rf_pipeline.joblib',
             Path('../Model') / 'xgb_pipeline.joblib'),
            (Path('/home/site/wwwroot/Model') / 'rf_pipeline.joblib',
             Path('/home/site/wwwroot/Model') / 'xgb_pipeline.joblib'),
            (Path(__file__).parent / 'Model' / 'rf_pipeline.joblib' if '__file__' in globals() else None,
             Path(__file__).parent / 'Model' / 'xgb_pipeline.joblib' if '__file__' in globals() else None),
        ]

        for rf_path, xgb_path in potential_model_paths:
            if rf_path is None or xgb_path is None:
                continue
            try:
                print(f"Trying model paths: {rf_path}, {xgb_path}")
                if rf_path.exists() and xgb_path.exists():
                    print(f"Found models at: {rf_path}, {xgb_path}")
                    self.random_forest_model = joblib.load(rf_path)
                    self.xgboost_model = joblib.load(xgb_path)
                    return True
            except Exception as e:
                print(f"  failed: {e}")

        print("No models found in any of the potential locations.")
        return False

    def _safe_image(self, filename, width, fallback_text):
        """Use pn.pane.Image if the file exists, otherwise a Markdown placeholder."""
        candidates = [
            Path(filename),
            Path('.') / filename,
            Path('/home/site/wwwroot') / filename,
        ]
        if '__file__' in globals():
            candidates.append(Path(__file__).parent / filename)

        for p in candidates:
            try:
                if p.exists():
                    print(f"Using image: {p}")
                    return pn.pane.Image(str(p), width=width)
            except Exception:
                pass

        print(f"Image not found: {filename} — using placeholder")
        return pn.pane.Markdown(
            fallback_text,
            width=width,
            styles={'background': '#f9f9f9', 'padding': '10px',
                    'border': '1px solid #ddd', 'font-size': '14px'}
        )

    def setup_components(self):
        self.feature_sliders = {}
        for feature, range_info in self.feature_ranges.items():
            base_label = self.feature_descriptions[feature]
            label_with_range = f"{base_label}   [range: {range_info['min']} to {range_info['max']}]"

            slider = pn.widgets.EditableFloatSlider(
                name=label_with_range,
                start=range_info['min'],
                end=range_info['max'],
                value=range_info['initial'],
                step=(range_info['max'] - range_info['min']) / 100,
                width=540,
                fixed_start=range_info['min'],
                fixed_end=range_info['max'],
                stylesheets=[WIDGET_CSS]
            )
            if self.models_loaded:
                slider.param.watch(self.update_predictions, 'value')
            self.feature_sliders[feature] = slider

        self.sensitivity_feature_selector = pn.widgets.Select(
            name='Select Feature (Environmental Variables) for Sensitivity Analysis',
            options=list(self.feature_descriptions.values()),
            value=list(self.feature_descriptions.values())[0],
            width=540,
            stylesheets=[WIDGET_CSS]
        )
        if self.models_loaded:
            self.sensitivity_feature_selector.param.watch(
                self.update_sensitivity_analysis_plot, 'value'
            )

    def create_layout_components(self):
        self.logo = self._safe_image(
            'Logo.png', 600,
            "# CHAB Prediction Dashboard"
        )
        self.map_image = self._safe_image(
            'Map.png', 500,
            "**Map image not found.** Expected `Map.png` in the working directory."
        )

        self.map_description = pn.pane.Markdown("""
Study area map demonstrating data collection locations within the Sacramento – San Joaquin Delta (Delta):  
(a) Selected 14 data collection locations where *Microcystis* (cells/ml) and other 14 environmental variables' data were collected between 2014 and 2019 within Delta and  
(b) Range of maximum values of *Microcystis* (cells/ml) throughout the Delta.  

Note: *Microcystis* (cells/ml) represents qPCR-based lab-analyzed toxigenic *Microcystis* (cells/ml)
            """, width=500, styles={'font-size': '14px'})

        # If models didn't load, show explanatory placeholders instead of empty panes
        if self.models_loaded:
            rf_initial = ""
            xgb_initial = ""
        else:
            rf_initial = ("### Random Forest Predictions\n\n"
                          "**Models did not load — predictions unavailable.** "
                          "Check the Azure log for the path that was tried.")
            xgb_initial = ("### XGBoost Predictions\n\n"
                           "**Models did not load — predictions unavailable.**")

        self.random_forest_markdown = pn.pane.Markdown(rf_initial, width=450, styles={'font-size': '15px'})
        self.xgboost_markdown = pn.pane.Markdown(xgb_initial, width=450, styles={'font-size': '15px'})
        self.model_comparison_plot = pn.pane.HoloViews(width=420)
        self.sensitivity_analysis_plot = pn.pane.HoloViews(width=750)

        self.last_update = pn.pane.Markdown(
            "<div style='text-align: left; color: #c0392b; font-style: italic; "
            "padding-left: 20px; font-weight: 600; font-size: 16px; margin-top: 20px;'>"
            "The last update of these models has been completed on May 26, 2026."
            "</div>",
            sizing_mode='stretch_width'
        )

        self.disclaimer = pn.pane.Markdown("""
---
**Disclaimer: This dashboard is still in beta.**

**Thank you for evaluating the CHAB Dashboard.**

**If you have feedback, suggestions or questions, please contact:**
- **Peyman Namadi (Peyman.Hosseinzadehnamadi@Water.ca.gov)**
- **Gourab Saha (gourab.saha@water.ca.gov)**
- **Kevin He (Kevin.He@Water.ca.gov)**
        """, styles={'font-size': '14px'})

    def update_predictions(self, event):
        if not self.models_loaded:
            return

        current_values = {f: s.value for f, s in self.feature_sliders.items()}
        feature_order = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4", "WaterTemp", "DO", "pH"]
        sample_df = pd.DataFrame([current_values])[feature_order]

        rf_label = self.random_forest_model.predict(sample_df)[0]
        rf_prob = self.random_forest_model.predict_proba(sample_df)[0][1]
        xgb_label = self.xgboost_model.predict(sample_df)[0]
        xgb_prob = self.xgboost_model.predict_proba(sample_df)[0][1]

        risky_text = "Risky CHAB event (*Microcystis* > 4,000 cells/ml)"
        safe_text  = "Not Risky CHAB event (*Microcystis* ≤ 4,000 cells/ml)"

        rf_result_text  = risky_text if rf_label == 1 else safe_text
        xgb_result_text = risky_text if xgb_label == 1 else safe_text

        self.random_forest_markdown.object = f"""
### Random Forest Predictions

**Result:** {rf_result_text}  
**Probability:** {rf_prob:.2%} chance of having Risky CHAB events
        """

        self.xgboost_markdown.object = f"""
### XGBoost Predictions

**Result:** {xgb_result_text}  
**Probability:** {xgb_prob:.2%} chance of having Risky CHAB events
        """

        comparison_df = pd.DataFrame({
            'Model': ['Random Forest', 'XGBoost'],
            'Probability': [rf_prob * 100, xgb_prob * 100]
        })

        self.model_comparison_plot.object = comparison_df.hvplot.bar(
            x='Model',
            y='Probability',
            title='',
            width=420,
            height=340,
            color='#1f77b4',
            ylabel='Probability of having\nRisky CHAB event (%)'
        ).opts(
            tools=['hover'],
            toolbar=None,
            fontsize={'title': 14, 'labels': 13, 'xticks': 12, 'yticks': 11},
            bar_width=0.4,
            ylim=(0, 100)
        )

        self.update_sensitivity_analysis_plot(None)

    def update_sensitivity_analysis_plot(self, event):
        if not self.models_loaded:
            return

        current_values = {f: s.value for f, s in self.feature_sliders.items()}
        selected_feature_desc = self.sensitivity_feature_selector.value
        selected_feature = next(k for k, v in self.feature_descriptions.items() if v == selected_feature_desc)
        range_info = self.feature_ranges[selected_feature]

        x_values = np.linspace(range_info['min'], range_info['max'], 20)
        rf_probs = []
        xgb_probs = []

        feature_order = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4", "WaterTemp", "DO", "pH"]
        for x_val in x_values:
            temp_values = current_values.copy()
            temp_values[selected_feature] = x_val
            temp_df = pd.DataFrame([temp_values])[feature_order]

            rf_probs.append(self.random_forest_model.predict_proba(temp_df)[0][1] * 100)
            xgb_probs.append(self.xgboost_model.predict_proba(temp_df)[0][1] * 100)

        sensitivity_df = pd.DataFrame({
            'Value': x_values,
            'Random Forest': rf_probs,
            'XGBoost': xgb_probs
        })

        self.sensitivity_analysis_plot.object = sensitivity_df.hvplot.line(
            x='Value',
            y=['Random Forest', 'XGBoost'],
            title=f'Sensitivity Analysis: {selected_feature_desc}',
            width=750,
            height=420,
            xlabel=selected_feature_desc,
            ylabel='Probability of having\nRisky CHAB event\n(Microcystis > 4,000 cells/ml) (%)',
            group_label='Models'
        ).opts(
            legend_position='bottom_right',
            tools=['hover'],
            toolbar=None,
            fontsize={'title': 14, 'labels': 13, 'xticks': 12, 'yticks': 11, 'legend': 12, 'legend_title': 13}
        )

    def create_dashboard_layout(self):
        input_header       = pn.pane.Markdown("## Input Features (Environmental Variables)",
                                              css_classes=['section-header'])
        results_header     = pn.pane.Markdown("## CHAB Toxicity Risk Predictions",
                                              css_classes=['section-header'])
        comparison_header  = pn.pane.Markdown("## Top Two Models' Comparison",
                                              css_classes=['section-header'])
        sensitivity_header = pn.pane.Markdown("## Sensitivity Analysis",
                                              css_classes=['section-header'])

        left_col = pn.Column(
            input_header,
            *[self.feature_sliders[feat] for feat in self.feature_ranges.keys()],
            sensitivity_header,
            self.sensitivity_feature_selector,
            self.sensitivity_analysis_plot,
            width=840,
            scroll=True
        )

        model_results = pn.Column(
            results_header,
            pn.Row(
                pn.Column(
                    self.random_forest_markdown,
                    self.xgboost_markdown,
                    width=450
                ),
                pn.Column(
                    self.map_image,
                    self.map_description,
                    width=500
                )
            ),
            comparison_header,
            self.model_comparison_plot,
            width=980
        )

        layout = pn.Column(
            pn.Row(self.logo, sizing_mode='stretch_width'),
            pn.Row(left_col, model_results),
            self.last_update,
            self.disclaimer,
            sizing_mode='stretch_width'
        )

        return layout


# ============================================================
# IMPORTANT: This block runs at MODULE IMPORT TIME.
# Azure's `panel serve` imports the module, so .servable() must
# be called here (NOT inside `if __name__ == "__main__":`).
# ============================================================
dashboard_app = ToxinPredictionDashboard()
dashboard = dashboard_app.create_dashboard_layout()
dashboard.servable()


# Optional: keep a local-run entry point too, but it's not what Azure uses.
if __name__ == "__main__":
    try:
        pn.serve(dashboard, show=True, port=5006)
    except Exception as e:
        print(f"Error serving dashboard locally: {e}")

==== DASHBOARD INITIALIZATION ====
Current working directory: e:\DWR\HAB\HAB-main\HAB-main\HAB-main
Directory contents: ['Data_2', 'environment.yml', 'habdashboard.ipynb', 'habdashboardv2.ipynb', 'HAB_Dashboard_9features_RFandXGB.ipynb', 'Logo.png', 'Map.png', 'Model', 'README.md', 'readme.pdf', 'run_server.sh', 'Testing_RF_XGB_for _reproducability_V3.ipynb']
Trying model paths: Model\rf_pipeline.joblib, Model\xgb_pipeline.joblib
Found models at: Model\rf_pipeline.joblib, Model\xgb_pipeline.joblib
Using image: Logo.png
Using image: Map.png
Launching server at http://localhost:5006
